In [3]:
# Import required modules
import sys
sys.path.append('..')

from src.fretboard_optimizer import (
    FretboardOptimizer, 
    MidiNote, 
    FretPosition,
    midi_number_to_note_name
)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

ModuleNotFoundError: No module named 'numpy'

## 1. Basic Usage: C Major Scale

In [ ]:
# Create a C major scale
melody = [
    MidiNote(60, 0.0, 0.5),   # C4
    MidiNote(62, 0.5, 1.0),   # D4
    MidiNote(64, 1.0, 1.5),   # E4
    MidiNote(65, 1.5, 2.0),   # F4
    MidiNote(67, 2.0, 2.5),   # G4
    MidiNote(69, 2.5, 3.0),   # A4
    MidiNote(71, 3.0, 3.5),   # B4
    MidiNote(72, 3.5, 4.0),   # C5
]

# Initialize optimizer
optimizer = FretboardOptimizer()

# Run optimization
path, tablature = optimizer.optimize(melody)

# Display results
print(tablature)

## 2. Exploring Possible Positions

Let's see all the ways we can play a single note on the guitar.

In [ ]:
# Find all positions for G4 (MIDI 67)
note = 67
positions = optimizer.get_possible_positions(note)

string_names = ['E (low)', 'A', 'D', 'G', 'B', 'E (high)']

print(f"G4 (MIDI {note}) can be played at {len(positions)} positions:\n")
for pos in positions:
    print(f"  String {pos.string + 1} ({string_names[pos.string]:>8}): Fret {pos.fret}")

## 3. Visualizing the Cost Function

Let's visualize how transition costs change with fret distance and string jumps.

In [ ]:
# Create cost matrix for different transitions
fret_distances = range(0, 10)
string_jumps = range(0, 6)

cost_matrix = np.zeros((len(string_jumps), len(fret_distances)))

for i, string_jump in enumerate(string_jumps):
    for j, fret_dist in enumerate(fret_distances):
        pos1 = FretPosition(0, 5, 45)
        pos2 = FretPosition(string_jump, 5 + fret_dist, 45 + fret_dist)
        cost_matrix[i, j] = optimizer.calculate_transition_cost(pos1, pos2)

# Plot heatmap
plt.figure(figsize=(12, 6))
plt.imshow(cost_matrix, aspect='auto', cmap='YlOrRd', interpolation='nearest')
plt.colorbar(label='Transition Cost')
plt.xlabel('Fret Distance')
plt.ylabel('String Jump')
plt.title('Transition Cost Heatmap\n(Yellow = Easy, Red = Difficult)')
plt.xticks(range(len(fret_distances)), fret_distances)
plt.yticks(range(len(string_jumps)), string_jumps)

# Add text annotations
for i in range(len(string_jumps)):
    for j in range(len(fret_distances)):
        text = plt.text(j, i, f'{cost_matrix[i, j]:.1f}',
                       ha="center", va="center", color="black", fontsize=8)

plt.tight_layout()
plt.show()

print("Notice how costs spike at fret distance > 4 (stretch penalty!)")

## 4. Comparing Different Melodies

Let's see how the optimizer handles different types of melodic patterns.

In [ ]:
# Stepwise melody (easy)
stepwise = [
    MidiNote(60 + i, i * 0.5, (i + 1) * 0.5) 
    for i in range(8)
]

# Melody with jumps (harder)
jumpy = [
    MidiNote(60, 0.0, 0.5),
    MidiNote(72, 0.5, 1.0),
    MidiNote(55, 1.0, 1.5),
    MidiNote(76, 1.5, 2.0),
    MidiNote(60, 2.0, 2.5),
]

# Optimize both
opt1 = FretboardOptimizer()
path1, _ = opt1.optimize(stepwise)

opt2 = FretboardOptimizer()
path2, _ = opt2.optimize(jumpy)

# Calculate statistics
def calc_stats(path):
    if len(path) < 2:
        return 0, 0
    fret_moves = [abs(path[i].fret - path[i+1].fret) for i in range(len(path)-1)]
    string_jumps = [abs(path[i].string - path[i+1].string) for i in range(len(path)-1)]
    return np.mean(fret_moves), np.mean(string_jumps)

avg_fret1, avg_string1 = calc_stats(path1)
avg_fret2, avg_string2 = calc_stats(path2)

print("Stepwise Melody:")
print(f"  Avg fret movement: {avg_fret1:.2f}")
print(f"  Avg string jumps:  {avg_string1:.2f}")

print("\nJumpy Melody:")
print(f"  Avg fret movement: {avg_fret2:.2f}")
print(f"  Avg string jumps:  {avg_string2:.2f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Stepwise
strings1 = [p.string for p in path1]
frets1 = [p.fret for p in path1]
ax1.plot(strings1, 'o-', label='String', markersize=8)
ax1.plot(frets1, 's-', label='Fret', markersize=8)
ax1.set_title('Stepwise Melody - Finger Position Over Time')
ax1.set_xlabel('Note Index')
ax1.set_ylabel('String / Fret Number')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Jumpy
strings2 = [p.string for p in path2]
frets2 = [p.fret for p in path2]
ax2.plot(strings2, 'o-', label='String', markersize=8)
ax2.plot(frets2, 's-', label='Fret', markersize=8)
ax2.set_title('Jumpy Melody - Finger Position Over Time')
ax2.set_xlabel('Note Index')
ax2.set_ylabel('String / Fret Number')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Custom Tunings

Let's try Drop-D tuning, popular in rock and metal.

In [ ]:
# Drop-D tuning: D2, A2, D3, G3, B3, E4
drop_d_tuning = [38, 45, 50, 55, 59, 64]

# D minor scale in drop-D
d_minor_scale = [
    MidiNote(62, 0.0, 0.5),   # D4
    MidiNote(64, 0.5, 1.0),   # E4
    MidiNote(65, 1.0, 1.5),   # F4
    MidiNote(67, 1.5, 2.0),   # G4
    MidiNote(69, 2.0, 2.5),   # A4
    MidiNote(70, 2.5, 3.0),   # A#4
    MidiNote(72, 3.0, 3.5),   # C5
    MidiNote(74, 3.5, 4.0),   # D5
]

# Compare standard vs drop-D
standard_opt = FretboardOptimizer()
dropd_opt = FretboardOptimizer(tuning=drop_d_tuning)

_, tab_standard = standard_opt.optimize(d_minor_scale)
_, tab_dropd = dropd_opt.optimize(d_minor_scale)

print("STANDARD TUNING (E-A-D-G-B-E):")
print("=" * 80)
print(tab_standard)

print("\n\nDROP-D TUNING (D-A-D-G-B-E):")
print("=" * 80)
print(tab_dropd)

## 6. Customizing Cost Parameters

Adjust the cost function to match your playing style!

In [ ]:
# Test melody
test_melody = [
    MidiNote(64, 0.0, 0.5),
    MidiNote(67, 0.5, 1.0),
    MidiNote(71, 1.0, 1.5),
    MidiNote(74, 1.5, 2.0),
    MidiNote(67, 2.0, 2.5),
]

# Default settings
opt_default = FretboardOptimizer()
path_default, _ = opt_default.optimize(test_melody)

# Prefer staying on same string (low string jump penalty)
opt_same_string = FretboardOptimizer()
opt_same_string.STRING_JUMP_WEIGHT = 0.1
path_same_string, _ = opt_same_string.optimize(test_melody)

# Prefer changing strings (high string jump penalty forces staying put)
opt_diff_string = FretboardOptimizer()
opt_diff_string.STRING_JUMP_WEIGHT = 5.0
path_diff_string, _ = opt_diff_string.optimize(test_melody)

# Display comparison
print("Default Settings:")
for i, pos in enumerate(path_default):
    print(f"  Note {i+1}: String {pos.string + 1}, Fret {pos.fret}")

print("\nLow String Jump Penalty (prefer same string):")
for i, pos in enumerate(path_same_string):
    print(f"  Note {i+1}: String {pos.string + 1}, Fret {pos.fret}")

print("\nHigh String Jump Penalty (prefer different strings):")
for i, pos in enumerate(path_diff_string):
    print(f"  Note {i+1}: String {pos.string + 1}, Fret {pos.fret}")

## 7. Graph Visualization

Let's visualize the actual graph structure for a small melody.

In [ ]:
# Simple 3-note melody
simple_melody = [
    MidiNote(60, 0.0, 0.5),  # C4
    MidiNote(64, 0.5, 1.0),  # E4
    MidiNote(67, 1.0, 1.5),  # G4
]

opt = FretboardOptimizer()
opt.build_graph(simple_melody)

print("Graph Statistics:")
print(f"  Total nodes: {len(opt.graph)}")
print(f"\nPositions per note:")
for note_idx, positions in opt.positions_by_note.items():
    note_name = midi_number_to_note_name(simple_melody[note_idx].midi_number)
    print(f"  {note_name}: {len(positions)} positions")
    for pos in positions:
        print(f"    - String {pos.string + 1}, Fret {pos.fret}")

# Count edges
total_edges = sum(len(neighbors) for neighbors in opt.graph.values())
print(f"\n  Total edges: {total_edges}")

## 8. Performance Analysis

How does performance scale with melody length?

In [ ]:
import time

melody_lengths = [5, 10, 20, 30, 50]
times = []

for length in melody_lengths:
    # Create random melody
    melody = [
        MidiNote(60 + (i % 12), i * 0.5, (i + 1) * 0.5)
        for i in range(length)
    ]
    
    # Time the optimization
    opt = FretboardOptimizer()
    start = time.time()
    opt.optimize(melody)
    elapsed = time.time() - start
    times.append(elapsed)
    
    print(f"Length {length:2d}: {elapsed*1000:.2f} ms")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(melody_lengths, times, 'o-', linewidth=2, markersize=8)
plt.xlabel('Melody Length (number of notes)')
plt.ylabel('Optimization Time (seconds)')
plt.title('Performance Scaling')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAverage time per note: {np.mean(times) / np.mean(melody_lengths) * 1000:.2f} ms")